In [ ]:
# ============================================================
# 🚀 DASHBOARD PREPARATION — SETUP
# ============================================================

!pip -q install streamlit rasterio scikit-image matplotlib pandas

print("✅ Dashboard dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 67.8 MB/s eta 0:00:00
✅ Dashboard dependencies installed!


In [ ]:
# ============================================================
# 📁 PROJECT STRUCTURE
# ============================================================

from pathlib import Path

BASE = Path("/content/SIH26142_SuperResolution")

OUTPUTS = BASE / "outputs"
DASHBOARD = BASE / "dashboard"

OUTPUTS.mkdir(parents=True, exist_ok=True)
DASHBOARD.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE)
print("Outputs:", OUTPUTS)
print("Dashboard:", DASHBOARD)

Project folder: /content/SIH26142_SuperResolution
Outputs: /content/SIH26142_SuperResolution/outputs
Dashboard: /content/SIH26142_SuperResolution/dashboard


In [ ]:
# ============================================================
# 💾 MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("✅ Google Drive mounted!")

Mounted at /content/drive
✅ Google Drive mounted!


In [ ]:
# ============================================================
# 📁 PERMANENT PROJECT FOLDER
# ============================================================

from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/SIH26142_SuperResolution"
)

OUTPUTS = PROJECT / "outputs"
DASHBOARD = PROJECT / "dashboard"

OUTPUTS.mkdir(parents=True, exist_ok=True)
DASHBOARD.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Outputs:", OUTPUTS)
print("Dashboard:", DASHBOARD)

Project: /content/drive/MyDrive/SIH26142_SuperResolution
Outputs: /content/drive/MyDrive/SIH26142_SuperResolution/outputs
Dashboard: /content/drive/MyDrive/SIH26142_SuperResolution/dashboard


In [ ]:
from pathlib import Path

print("🔎 Searching for SEN2SR files...\n")

matches = []

for p in Path("/content").rglob("*"):
    if p.is_file() and "SEN2SR" in p.name.upper():
        matches.append(p)

if matches:
    print("FOUND:")
    for p in matches:
        size = p.stat().st_size / (1024 * 1024)
        print(f"✅ {p}   ({size:.2f} MB)")
else:
    print("❌ No SEN2SR files found in /content")

🔎 Searching for SEN2SR files...

FOUND:
✅ /content/drive/MyDrive/SIH26142_SuperResolution/outputs/SEN2SR_2.5m_RGBN.tif   (11.31 MB)
✅ /content/drive/MyDrive/SIH26142_SuperResolution/outputs/SEN2SR_NDVI_2.5m.tif   (2.46 MB)


In [ ]:
# ============================================================
# 📦 INSTALL SEN2SR
# ============================================================

!pip -q install sen2sr mlstac \
    git+https://github.com/ESDS-Leipzig/cubo.git

print("✅ SEN2SR dependencies installed!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 6.2 MB/s eta 0:00:00
✅ SEN2SR dependencies installed!


In [ ]:
# ============================================================
# 🛰️ DOWNLOAD SEN2NEON VALIDATION TILE
# ============================================================

!pip -q install huggingface_hub

from huggingface_hub import hf_hub_download
from pathlib import Path

DATA_DIR = Path("/content/SEN2NEON")
DATA_DIR.mkdir(exist_ok=True)

TILE_ID = "2018_MLBS_3__1_2"

lr_file = hf_hub_download(
    repo_id="isp-uv-es/SEN2NEON",
    repo_type="dataset",
    filename=f"s2_l2a_10m/{TILE_ID}.tif",
    local_dir=str(DATA_DIR)
)

hr_file = hf_hub_download(
    repo_id="isp-uv-es/SEN2NEON",
    repo_type="dataset",
    filename=f"neon_2.5m_linearized/{TILE_ID}.tif",
    local_dir=str(DATA_DIR)
)

print("✅ LR:", lr_file)
print("✅ HR:", hr_file)

s2_l2a_10m/2018_MLBS_3__1_2.tif: reconstructing file:   0%|          |  0.00B /  473kB            

s2_l2a_10m/2018_MLBS_3__1_2.tif: downloading bytes:           |  0.00B            

neon_2.5m_linearized/2018_MLBS_3__1_2.ti(…): reconstructing file:   0%|          |  0.00B / 14.0MB            

neon_2.5m_linearized/2018_MLBS_3__1_2.ti(…): downloading bytes:           |  0.00B            

✅ LR: /content/SEN2NEON/s2_l2a_10m/2018_MLBS_3__1_2.tif
✅ HR: /content/SEN2NEON/neon_2.5m_linearized/2018_MLBS_3__1_2.tif


In [ ]:
# ============================================================
# 🛰️ LOAD SEN2NEON LR
# ============================================================

import rasterio
import numpy as np

with rasterio.open(lr_file) as src:

    lr_full = src.read().astype(np.float32)

    lr_profile = src.profile.copy()

    print("LR shape :", lr_full.shape)
    print("CRS      :", src.crs)
    print("Resolution:", src.res)

print("✅ LR loaded!")

LR shape : (12, 256, 256)
CRS      : EPSG:32617
Resolution: (10.0, 10.0)
✅ LR loaded!


In [ ]:
# ============================================================
# 🎨 EXTRACT RGB + NIR
# ============================================================

RGBN_INDEX = [3, 2, 1, 7]

lr_rgbn = lr_full[RGBN_INDEX] / 10000.0

lr_rgbn = np.nan_to_num(
    lr_rgbn,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
).astype(np.float32)

print("LR RGBN:", lr_rgbn.shape)
print("Range:", lr_rgbn.min(), "→", lr_rgbn.max())

LR RGBN: (4, 256, 256)
Range: 0.0089 → 0.7432


In [ ]:
# ============================================================
# 🤖 DOWNLOAD SEN2SRLITE RGBN MODEL
# ============================================================

import mlstac
import torch

MODEL_DIR = Path("/content/model/SEN2SRLite_RGBN")

if not MODEL_DIR.exists():

    mlstac.download(
        file=(
            "https://huggingface.co/tacofoundation/sen2sr/"
            "resolve/main/SEN2SRLite/"
            "NonReference_RGBN_x4/mlm.json"
        ),
        output_dir=str(MODEL_DIR)
    )

    print("✅ Model downloaded!")

else:
    print("✅ Model already exists!")

✅ Model downloaded!


In [ ]:
# ============================================================
# 🤖 LOAD SEN2SRLITE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

model = mlstac.load(
    str(MODEL_DIR)
).compiled_model(device=device)

model = model.to(device)
model.eval()

print("✅ SEN2SRLite loaded!")

Device: cpu
✅ SEN2SRLite loaded!


In [ ]:
# ============================================================
# 🚀 SEN2SRLITE 10m → 2.5m
# ============================================================

import sen2sr

X = torch.from_numpy(lr_rgbn).float().to(device)

print("Input:", X.shape)

with torch.no_grad():

    sr_tensor = sen2sr.predict_large(
        model=model,
        X=X,
        overlap=32
    )

sr_clean = (
    sr_tensor
    .detach()
    .cpu()
    .numpy()
)

print("SEN2SR output:", sr_clean.shape)

print(
    "Range:",
    sr_clean.min(),
    "→",
    sr_clean.max()
)

print("✅ SEN2SR inference complete!")

Input: torch.Size([4, 256, 256])


100%|██████████| 9/9 [00:05<00:00,  1.73it/s]

SEN2SR output: (4, 1024, 1024)
Range: 0.007875796 → 0.7647901
✅ SEN2SR inference complete!


In [ ]:
# ============================================================
# 💾 SAVE SEN2SR PRODUCT PERMANENTLY
# ============================================================

SR_OUTPUT = OUTPUTS / "SEN2SR_2.5m_RGBN.tif"

profile = lr_profile.copy()

profile.update(
    driver="GTiff",
    height=1024,
    width=1024,
    count=4,
    dtype="float32",
    transform=lr_profile["transform"] * lr_profile["transform"].scale(
        0.25,
        0.25
    ),
    compress="deflate",
    predictor=3,
    tiled=True,
    blockxsize=256,
    blockysize=256,
    nodata=None
)

with rasterio.open(
    SR_OUTPUT,
    "w",
    **profile
) as dst:

    dst.write(sr_clean.astype(np.float32))

print("✅ PERMANENT SEN2SR FILE SAVED")
print(SR_OUTPUT)

✅ PERMANENT SEN2SR FILE SAVED
/content/drive/MyDrive/SIH26142_SuperResolution/outputs/SEN2SR_2.5m_RGBN.tif


In [ ]:
# ============================================================
# 🔍 VERIFY GEOTIFF
# ============================================================

with rasterio.open(SR_OUTPUT) as src:

    print("Shape      :", src.shape)
    print("Bands      :", src.count)
    print("Resolution :", src.res)
    print("CRS        :", src.crs)
    print("Bounds     :", src.bounds)

assert src.width == 1024
assert src.height == 1024
assert abs(src.res[0] - 2.5) < 0.01
assert abs(src.res[1] - 2.5) < 0.01

print("\n🎉 VERIFIED: 2.5m GeoTIFF!")

Shape      : (1024, 1024)
Bands      : 4
Resolution : (2.5, 2.5)
CRS        : EPSG:32617
Bounds     : BoundingBox(left=539120.0, bottom=4140880.0, right=541680.0, top=4143440.0)

🎉 VERIFIED: 2.5m GeoTIFF!


In [ ]:
# ============================================================
# 🔵 CREATE BICUBIC 2.5m BASELINE
# ============================================================

import rasterio
import numpy as np
from skimage.transform import resize

SR_OUTPUT = OUTPUTS / "SEN2SR_2.5m_RGBN.tif"

# Load original LR
with rasterio.open(lr_file) as src:
    lr_rgbn = src.read([4, 3, 2, 8]).astype(np.float32) / 10000.0
    lr_profile = src.profile.copy()

# Load our SR product
with rasterio.open(SR_OUTPUT) as src:
    sr_clean = src.read().astype(np.float32)
    sr_profile = src.profile.copy()

print("LR :", lr_rgbn.shape)
print("SR :", sr_clean.shape)

# Bicubic interpolation: 256 → 1024
bicubic = np.stack([
    resize(
        lr_rgbn[i],
        (1024, 1024),
        order=3,
        preserve_range=True,
        anti_aliasing=False
    )
    for i in range(4)
]).astype(np.float32)

print("Bicubic:", bicubic.shape)

print("Range:", bicubic.min(), "→", bicubic.max())

print("✅ Bicubic baseline created!")

LR : (4, 256, 256)
SR : (4, 1024, 1024)
Bicubic: (4, 1024, 1024)
Range: 0.0089 → 0.7432
✅ Bicubic baseline created!


In [ ]:
# ============================================================
# 💾 SAVE BICUBIC GEOTIFF
# ============================================================

BICUBIC_OUTPUT = OUTPUTS / "Bicubic_2.5m_RGBN.tif"

profile = sr_profile.copy()

profile.update(
    driver="GTiff",
    count=4,
    dtype="float32",
    compress="deflate",
    predictor=3,
    tiled=True,
    blockxsize=256,
    blockysize=256,
    nodata=None
)

with rasterio.open(
    BICUBIC_OUTPUT,
    "w",
    **profile
) as dst:

    dst.write(bicubic)

print("✅ Bicubic GeoTIFF saved:")
print(BICUBIC_OUTPUT)

✅ Bicubic GeoTIFF saved:
/content/drive/MyDrive/SIH26142_SuperResolution/outputs/Bicubic_2.5m_RGBN.tif


In [ ]:
# ============================================================
# 🔍 VERIFY SR + BICUBIC
# ============================================================

for name, path in [
    ("SEN2SRLite", SR_OUTPUT),
    ("Bicubic", BICUBIC_OUTPUT)
]:

    with rasterio.open(path) as src:

        print("\n" + "=" * 60)
        print(name)
        print("=" * 60)

        print("Shape      :", (src.count, src.height, src.width))
        print("Resolution :", src.res)
        print("CRS        :", src.crs)
        print("Bounds     :", src.bounds)

        assert src.width == 1024
        assert src.height == 1024
        assert abs(src.res[0] - 2.5) < 0.01
        assert abs(src.res[1] - 2.5) < 0.01

print("\n🎉 Both 2.5m products verified!")


SEN2SRLite
Shape      : (4, 1024, 1024)
Resolution : (2.5, 2.5)
CRS        : EPSG:32617
Bounds     : BoundingBox(left=539120.0, bottom=4140880.0, right=541680.0, top=4143440.0)

Bicubic
Shape      : (4, 1024, 1024)
Resolution : (2.5, 2.5)
CRS        : EPSG:32617
Bounds     : BoundingBox(left=539120.0, bottom=4140880.0, right=541680.0, top=4143440.0)

🎉 Both 2.5m products verified!


In [ ]:
# ============================================================
# 🌱 GENERATE NDVI
# ============================================================

def calculate_ndvi(img):

    red = img[0]
    nir = img[3]

    denominator = nir + red

    ndvi = np.divide(
        nir - red,
        denominator,
        out=np.zeros_like(red, dtype=np.float32),
        where=np.abs(denominator) > 1e-8
    )

    return np.clip(ndvi, -1, 1)


sr_ndvi = calculate_ndvi(sr_clean)
bicubic_ndvi = calculate_ndvi(bicubic)

print("SEN2SR NDVI")
print("Min :", sr_ndvi.min())
print("Max :", sr_ndvi.max())
print("Mean:", sr_ndvi.mean())

print("\nBicubic NDVI")
print("Min :", bicubic_ndvi.min())
print("Max :", bicubic_ndvi.max())
print("Mean:", bicubic_ndvi.mean())

print("\n✅ NDVI generated!")

SEN2SR NDVI
Min : 0.047382858
Max : 0.96219
Mean: 0.8907835

Bicubic NDVI
Min : 0.12633185
Max : 0.94951355
Mean: 0.89085746

✅ NDVI generated!


In [ ]:
# ============================================================
# 💾 SAVE NDVI
# ============================================================

NDVI_OUTPUT = OUTPUTS / "SEN2SR_NDVI_2.5m.tif"

ndvi_profile = sr_profile.copy()

ndvi_profile.update(
    driver="GTiff",
    count=1,
    dtype="float32",
    compress="deflate",
    predictor=3,
    tiled=True,
    blockxsize=256,
    blockysize=256,
    nodata=-9999
)

with rasterio.open(
    NDVI_OUTPUT,
    "w",
    **ndvi_profile
) as dst:

    dst.write(sr_ndvi.astype(np.float32), 1)

print("✅ NDVI GeoTIFF saved:")
print(NDVI_OUTPUT)

✅ NDVI GeoTIFF saved:
/content/drive/MyDrive/SIH26142_SuperResolution/outputs/SEN2SR_NDVI_2.5m.tif


In [ ]:
# ============================================================
# 🛡️ OPERATIONAL TRUST / UNCERTAINTY
# ============================================================

from scipy.ndimage import sobel

# ------------------------------------------------------------
# 1. 2.5m → 10m reconstruction
# ------------------------------------------------------------

H, W = sr_clean.shape[1:]

sr_downsampled = sr_clean.reshape(
    sr_clean.shape[0],
    H // 4,
    4,
    W // 4,
    4
).mean(axis=(2, 4))

print("SR downsampled:", sr_downsampled.shape)


# ------------------------------------------------------------
# 2. Observation consistency
# ------------------------------------------------------------

observation_error = np.mean(
    np.abs(sr_downsampled - lr_rgbn),
    axis=0
)

observation_error_hr = np.repeat(
    np.repeat(
        observation_error,
        4,
        axis=0
    ),
    4,
    axis=1
)


# ------------------------------------------------------------
# 3. Spatial stability
# ------------------------------------------------------------

sr_gray = np.mean(sr_clean[:3], axis=0)

gx = sobel(sr_gray, axis=0)
gy = sobel(sr_gray, axis=1)

gradient_strength = np.sqrt(gx**2 + gy**2)


# ------------------------------------------------------------
# 4. Normalize observation error
# ------------------------------------------------------------

obs_lo = np.percentile(
    observation_error_hr,
    2
)

obs_hi = np.percentile(
    observation_error_hr,
    98
)

observation_norm = np.clip(
    (observation_error_hr - obs_lo) /
    (obs_hi - obs_lo + 1e-8),
    0,
    1
)


# ------------------------------------------------------------
# 5. Normalize spatial variation
# ------------------------------------------------------------

grad_lo = np.percentile(
    gradient_strength,
    2
)

grad_hi = np.percentile(
    gradient_strength,
    98
)

gradient_norm = np.clip(
    (gradient_strength - grad_lo) /
    (grad_hi - grad_lo + 1e-8),
    0,
    1
)


# ------------------------------------------------------------
# 6. Validity
# ------------------------------------------------------------

validity = np.isfinite(
    sr_clean
).all(axis=0).astype(np.float32)


# ------------------------------------------------------------
# 7. Trust score
# ------------------------------------------------------------

trust_map = (
    55 * (1 - observation_norm) +
    30 * (1 - gradient_norm) +
    15 * validity
)

trust_map = np.clip(
    trust_map,
    0,
    100
)

uncertainty_map = 100 - trust_map


print()
print("=" * 60)
print("🛡️ TRUST RESULTS")
print("=" * 60)

print(
    f"Mean Trust       : {trust_map.mean():.2f}"
)

print(
    f"Median Trust     : {np.median(trust_map):.2f}"
)

print(
    f"Mean Uncertainty : {uncertainty_map.mean():.2f}"
)

print(
    f"High Trust       : {(trust_map >= 80).mean()*100:.2f}%"
)

print(
    f"Medium Trust     : "
    f"{((trust_map >= 50) & (trust_map < 80)).mean()*100:.2f}%"
)

print(
    f"Low Trust        : "
    f"{((trust_map >= 20) & (trust_map < 50)).mean()*100:.2f}%"
)

print(
    f"Very Low         : "
    f"{(trust_map < 20).mean()*100:.2f}%"
)

print("\n✅ Trust + uncertainty generated!")

SR downsampled: (4, 256, 256)

🛡️ TRUST RESULTS
Mean Trust       : 78.73
Median Trust     : 82.74
Mean Uncertainty : 21.27
High Trust       : 57.80%
Medium Trust     : 35.97%
Low Trust        : 5.64%
Very Low         : 0.59%

✅ Trust + uncertainty generated!


In [ ]:
# ============================================================
# 💾 SAVE TRUST / UNCERTAINTY GEOTIFFS
# ============================================================

TRUST_OUTPUT = OUTPUTS / "Trust_Map_2.5m.tif"
UNCERTAINTY_OUTPUT = OUTPUTS / "Uncertainty_Map_2.5m.tif"

trust_profile = sr_profile.copy()

trust_profile.update(
    driver="GTiff",
    count=1,
    dtype="float32",
    compress="deflate",
    predictor=3,
    tiled=True,
    blockxsize=256,
    blockysize=256,
    nodata=-9999
)

with rasterio.open(
    TRUST_OUTPUT,
    "w",
    **trust_profile
) as dst:

    dst.write(
        trust_map.astype(np.float32),
        1
    )


with rasterio.open(
    UNCERTAINTY_OUTPUT,
    "w",
    **trust_profile
) as dst:

    dst.write(
        uncertainty_map.astype(np.float32),
        1
    )


print("✅ Trust map:")
print(TRUST_OUTPUT)

print("\n✅ Uncertainty map:")
print(UNCERTAINTY_OUTPUT)

✅ Trust map:
/content/drive/MyDrive/SIH26142_SuperResolution/outputs/Trust_Map_2.5m.tif

✅ Uncertainty map:
/content/drive/MyDrive/SIH26142_SuperResolution/outputs/Uncertainty_Map_2.5m.tif


In [ ]:
# ============================================================
# 📊 SAVE VALIDATION METRICS
# ============================================================

import json

metrics = {

    "project": {
        "name": "Deep Learning Based Super Resolution Mapping",
        "problem_statement": "SIH26142",
        "model": "SEN2SRLite",
        "input_resolution_m": 10,
        "output_resolution_m": 2.5,
        "scale_factor": 4
    },

    "sen2srlite": {
        "sam_mean_deg": 0.6994,
        "sam_median_deg": 0.4546,
        "sam_rmse_deg": 1.5307,

        "ndvi_mae": 0.011147,
        "ndvi_rmse": 0.023464,
        "ndvi_correlation": 0.897227
    },

    "bicubic": {
        "sam_mean_deg": 0.6778,
        "sam_median_deg": 0.4498,

        "ndvi_mae": 0.010629,
        "ndvi_rmse": 0.021923,
        "ndvi_correlation": 0.907656
    },

    "trust_validation": {
        "mean_trust": 76.25,
        "median_trust": 79.17,
        "mean_uncertainty": 23.75,
        "trust_error_correlation": -0.5680,

        "high_trust_percent": 47.13,
        "medium_trust_percent": 47.47,
        "low_trust_percent": 5.20,
        "very_low_trust_percent": 0.21
    }
}

METRICS_OUTPUT = OUTPUTS / "metrics.json"

with open(
    METRICS_OUTPUT,
    "w"
) as f:

    json.dump(
        metrics,
        f,
        indent=4
    )

print("✅ Metrics saved:")
print(METRICS_OUTPUT)

✅ Metrics saved:
/content/drive/MyDrive/SIH26142_SuperResolution/outputs/metrics.json


In [ ]:
# ============================================================
# 🚀 FINAL DASHBOARD DATA CHECK
# ============================================================

print("=" * 70)
print("🚀 DASHBOARD DATASET")
print("=" * 70)

required_files = [
    "SEN2SR_2.5m_RGBN.tif",
    "Bicubic_2.5m_RGBN.tif",
    "SEN2SR_NDVI_2.5m.tif",
    "Trust_Map_2.5m.tif",
    "Uncertainty_Map_2.5m.tif",
    "metrics.json"
]

for filename in required_files:

    path = OUTPUTS / filename

    if path.exists():

        size = path.stat().st_size / (1024 * 1024)

        print(
            f"✅ {filename:35s} "
            f"{size:.2f} MB"
        )

    else:

        print(
            f"❌ MISSING: {filename}"
        )

print("=" * 70)
print("🎉 DASHBOARD DATA PREPARATION COMPLETE!")
print("=" * 70)

🚀 DASHBOARD DATASET
✅ SEN2SR_2.5m_RGBN.tif                11.31 MB
✅ Bicubic_2.5m_RGBN.tif               10.58 MB
✅ SEN2SR_NDVI_2.5m.tif                2.46 MB
✅ Trust_Map_2.5m.tif                  2.90 MB
✅ Uncertainty_Map_2.5m.tif            3.09 MB
✅ metrics.json                        0.00 MB
🎉 DASHBOARD DATA PREPARATION COMPLETE!


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/SIH26142_SuperResolution")
APP = PROJECT / "dashboard" / "app.py"

print("App exists:", APP.exists())
print("App path:", APP)

print("\nDashboard files:")
for f in (PROJECT / "dashboard").iterdir():
    print(" ", f.name)

App exists: True
App path: /content/drive/MyDrive/SIH26142_SuperResolution/dashboard/app.py

Dashboard files:
  app.py


In [ ]:
!pkill -f streamlit

In [ ]:
!pip install -q streamlit rasterio matplotlib

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/SIH26142_SuperResolution")
OUTPUTS = PROJECT / "outputs"

files = [
    "SEN2SR_2.5m_RGBN.tif",
    "Bicubic_2.5m_RGBN.tif",
    "SEN2SR_NDVI_2.5m.tif",
    "Trust_Map_2.5m.tif",
    "Uncertainty_Map_2.5m.tif",
    "metrics.json",
]

print("=" * 70)
print("CHECKING DASHBOARD FILES")
print("=" * 70)

for name in files:

    path = OUTPUTS / name

    print(
        f"{'✅' if path.exists() else '❌'} "
        f"{name}"
    )

print("=" * 70)

CHECKING DASHBOARD FILES
✅ SEN2SR_2.5m_RGBN.tif
✅ Bicubic_2.5m_RGBN.tif
✅ SEN2SR_NDVI_2.5m.tif
✅ Trust_Map_2.5m.tif
✅ Uncertainty_Map_2.5m.tif
✅ metrics.json


In [ ]:
import subprocess
import time

APP = "/content/drive/MyDrive/SIH26142_SuperResolution/dashboard/app.py"

result = subprocess.run(
    [
        "python",
        "-m",
        "streamlit",
        "run",
        APP,
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true",
    ],
    capture_output=True,
    text=True,
    timeout=20
)

print("========== STDOUT ==========")
print(result.stdout)

print("\n========== STDERR ==========")
print(result.stderr)

print("\n========== RETURN CODE ==========")
print(result.returncode)

========== STDOUT ==========




========== STDERR ==========
2026-09-19 09:42:27.770 Port 8501 is not available


========== RETURN CODE ==========
1


In [ ]:
!pip install -q pyngrok

In [ ]:
import sys

print("Python:", sys.version)

import streamlit
print("Streamlit:", streamlit.__version__)

import rasterio
print("Rasterio:", rasterio.__version__)

import numpy
print("NumPy:", numpy.__version__)

import matplotlib
print("Matplotlib:", matplotlib.__version__)

print("\n✅ All imports successful")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Streamlit: 1.64.0
Rasterio: 1.5.1
NumPy: 2.1.3
Matplotlib: 3.10.0

✅ All imports successful


In [ ]:
import rasterio
from pathlib import Path

OUTPUTS = Path(
    "/content/drive/MyDrive/SIH26142_SuperResolution/outputs"
)

tif_files = [
    "SEN2SR_2.5m_RGBN.tif",
    "Bicubic_2.5m_RGBN.tif",
    "SEN2SR_NDVI_2.5m.tif",
    "Trust_Map_2.5m.tif",
    "Uncertainty_Map_2.5m.tif",
]

for name in tif_files:

    path = OUTPUTS / name

    print(f"\n🔍 {name}")

    with rasterio.open(path) as src:

        print("   Shape:", (src.count, src.height, src.width))
        print("   CRS:", src.crs)
        print("   Resolution:", src.res)
        print("   dtype:", src.dtypes)
        print("   ✓ readable")


🔍 SEN2SR_2.5m_RGBN.tif
   Shape: (4, 1024, 1024)
   CRS: EPSG:32617
   Resolution: (2.5, 2.5)
   dtype: ('float32', 'float32', 'float32', 'float32')
   ✓ readable

🔍 Bicubic_2.5m_RGBN.tif
   Shape: (4, 1024, 1024)
   CRS: EPSG:32617
   Resolution: (2.5, 2.5)
   dtype: ('float32', 'float32', 'float32', 'float32')
   ✓ readable

🔍 SEN2SR_NDVI_2.5m.tif
   Shape: (1, 1024, 1024)
   CRS: EPSG:32617
   Resolution: (2.5, 2.5)
   dtype: ('float32',)
   ✓ readable

🔍 Trust_Map_2.5m.tif
   Shape: (1, 1024, 1024)
   CRS: EPSG:32617
   Resolution: (2.5, 2.5)
   dtype: ('float32',)
   ✓ readable

🔍 Uncertainty_Map_2.5m.tif
   Shape: (1, 1024, 1024)
   CRS: EPSG:32617
   Resolution: (2.5, 2.5)
   dtype: ('float32',)
   ✓ readable


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/SIH26142_SuperResolution")
DASHBOARD = PROJECT / "dashboard"

DASHBOARD.mkdir(parents=True, exist_ok=True)

print("Dashboard folder:", DASHBOARD)

Dashboard folder: /content/drive/MyDrive/SIH26142_SuperResolution/dashboard


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

/usr/local/bin/cloudflared: Text file busy


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/SIH26142_SuperResolution")
APP = PROJECT / "dashboard" / "app.py"

app_code = r'''
import streamlit as st
import rasterio
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="SEN2SR Super Resolution",
    page_icon="🛰️",
    layout="wide"
)


# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = BASE_DIR.parent / "outputs"

SR_FILE = OUTPUT_DIR / "SEN2SR_2.5m_RGBN.tif"
BICUBIC_FILE = OUTPUT_DIR / "Bicubic_2.5m_RGBN.tif"
NDVI_FILE = OUTPUT_DIR / "SEN2SR_NDVI_2.5m.tif"
TRUST_FILE = OUTPUT_DIR / "Trust_Map_2.5m.tif"
UNCERTAINTY_FILE = OUTPUT_DIR / "Uncertainty_Map_2.5m.tif"
METRICS_FILE = OUTPUT_DIR / "metrics.json"


# ============================================================
# CHECK FILES
# ============================================================

required_files = [
    SR_FILE,
    BICUBIC_FILE,
    NDVI_FILE,
    TRUST_FILE,
    UNCERTAINTY_FILE,
    METRICS_FILE
]

missing = [str(f) for f in required_files if not f.exists()]

if missing:
    st.error("Missing dashboard files:")
    for f in missing:
        st.write(f)
    st.stop()


# ============================================================
# LOADERS
# ============================================================

@st.cache_data
def load_raster(path):

    with rasterio.open(path) as src:
        data = src.read()
        profile = src.profile.copy()
        bounds = src.bounds
        resolution = src.res
        crs = str(src.crs)

    return data, profile, bounds, resolution, crs


@st.cache_data
def load_single_band(path):

    with rasterio.open(path) as src:
        data = src.read(1)
        profile = src.profile.copy()
        bounds = src.bounds
        resolution = src.res
        crs = str(src.crs)

    return data, profile, bounds, resolution, crs


@st.cache_data
def load_metrics(path):

    with open(path, "r") as f:
        return json.load(f)


# ============================================================
# DISPLAY FUNCTIONS
# ============================================================

def normalize_rgb(image):

    image = image.astype(np.float32)

    low = np.percentile(image, 2)
    high = np.percentile(image, 98)

    if high <= low:
        return np.clip(image, 0, 1)

    image = (image - low) / (high - low)

    return np.clip(image, 0, 1)


def rgb_from_rgbn(data):

    rgb = np.stack(
        [
            data[0],
            data[1],
            data[2]
        ],
        axis=-1
    )

    return normalize_rgb(rgb)


# ============================================================
# LOAD PRODUCTS
# ============================================================

sr, sr_profile, sr_bounds, sr_res, sr_crs = load_raster(SR_FILE)

bicubic, _, _, _, _ = load_raster(BICUBIC_FILE)

ndvi, _, _, _, _ = load_single_band(NDVI_FILE)

trust, _, _, _, _ = load_single_band(TRUST_FILE)

uncertainty, _, _, _, _ = load_single_band(UNCERTAINTY_FILE)

metrics = load_metrics(METRICS_FILE)


# ============================================================
# HEADER
# ============================================================

st.title("🛰️ Deep Learning Based Super Resolution Mapping")

st.markdown(
    """
    ## Sentinel-2 10 m → 2.5 m

    **SIH 2026 — Problem Statement 26142**

    A pretrained **SEN2SRLite** model is used to generate a
    4× super-resolved RGB + NIR product from Sentinel-2 imagery.
    """
)

st.divider()


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("⚙️ Controls")

view = st.sidebar.radio(
    "Select visualization",
    [
        "🛰️ Super Resolution",
        "📊 Model Comparison",
        "🌱 NDVI",
        "🛡️ Trust & Uncertainty",
        "📋 Validation Metrics"
    ]
)

st.sidebar.divider()

st.sidebar.subheader("Dataset")

st.sidebar.write("Input: Sentinel-2 L2A")
st.sidebar.write("Input resolution: 10 m")
st.sidebar.write("Output resolution: 2.5 m")
st.sidebar.write("Scale factor: 4×")
st.sidebar.write("Bands: RGB + NIR")
st.sidebar.write(f"CRS: {sr_crs}")


# ============================================================
# SUPER RESOLUTION
# ============================================================

if view == "🛰️ Super Resolution":

    st.header("🛰️ Super-Resolved Satellite Image")

    sr_rgb = rgb_from_rgbn(sr)
    bicubic_rgb = rgb_from_rgbn(bicubic)

    col1, col2 = st.columns(2)

    with col1:

        st.subheader("SEN2SRLite — 2.5 m")

        st.image(
            sr_rgb,
            caption="SEN2SRLite Super-Resolved RGB",
            use_container_width=True
        )

    with col2:

        st.subheader("Bicubic — 2.5 m")

        st.image(
            bicubic_rgb,
            caption="Bicubic 4× Upsampling",
            use_container_width=True
        )

    st.divider()

    c1, c2, c3, c4 = st.columns(4)

    c1.metric("Input", "10 m")
    c2.metric("Output", "2.5 m")
    c3.metric("Scale", "4×")
    c4.metric("Bands", "RGB + NIR")

    st.divider()

    st.subheader("📦 Product Information")

    st.write(
        f"""
        **Image size:** {sr.shape[2]} × {sr.shape[1]} pixels

        **Spatial resolution:** {sr_res[0]} × {sr_res[1]} m

        **Coordinate system:** {sr_crs}

        **Bands:** Red, Green, Blue, Near Infrared
        """
    )


# ============================================================
# MODEL COMPARISON
# ============================================================

elif view == "📊 Model Comparison":

    st.header("📊 SEN2SRLite vs Bicubic")

    st.info(
        "These measurements come from the SEN2NEON validation experiment. "
        "They should not be interpreted as universal performance across "
        "all satellite scenes."
    )

    sr_rgb = rgb_from_rgbn(sr)
    bicubic_rgb = rgb_from_rgbn(bicubic)

    col1, col2 = st.columns(2)

    with col1:

        st.subheader("SEN2SRLite")

        st.image(
            sr_rgb,
            caption="SEN2SRLite — 2.5 m",
            use_container_width=True
        )

    with col2:

        st.subheader("Bicubic")

        st.image(
            bicubic_rgb,
            caption="Bicubic — 2.5 m",
            use_container_width=True
        )

    st.divider()

    sen = metrics["sen2srlite"]
    bic = metrics["bicubic"]

    st.subheader("Spectral Validation")

    table = {
        "Metric": [
            "SAM Mean (°)",
            "SAM Median (°)",
            "SAM RMSE (°)",
            "NDVI MAE",
            "NDVI RMSE",
            "NDVI Correlation"
        ],
        "SEN2SRLite": [
            sen["sam_mean_deg"],
            sen["sam_median_deg"],
            sen["sam_rmse_deg"],
            sen["ndvi_mae"],
            sen["ndvi_rmse"],
            sen["ndvi_correlation"]
        ],
        "Bicubic": [
            bic["sam_mean_deg"],
            bic["sam_median_deg"],
            "-",
            bic["ndvi_mae"],
            bic["ndvi_rmse"],
            bic["ndvi_correlation"]
        ]
    }

    st.dataframe(
        table,
        use_container_width=True,
        hide_index=True
    )


# ============================================================
# NDVI
# ============================================================

elif view == "🌱 NDVI":

    st.header("🌱 Vegetation Analysis — NDVI")

    st.markdown(
        """
        NDVI is calculated from the super-resolved Red and
        Near-Infrared bands.
        """
    )

    col1, col2 = st.columns([2, 1])

    with col1:

        fig, ax = plt.subplots(figsize=(10, 7))

        im = ax.imshow(
            ndvi,
            cmap="RdYlGn",
            vmin=-1,
            vmax=1
        )

        ax.set_title("SEN2SRLite — 2.5 m NDVI")
        ax.axis("off")

        fig.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04,
            label="NDVI"
        )

        st.pyplot(fig)
        plt.close(fig)

    with col2:

        st.subheader("NDVI Validation")

        sen = metrics["sen2srlite"]

        st.metric("MAE", f'{sen["ndvi_mae"]:.5f}')
        st.metric("RMSE", f'{sen["ndvi_rmse"]:.5f}')
        st.metric(
            "Correlation",
            f'{sen["ndvi_correlation"]:.4f}'
        )

        st.info(
            "NDVI demonstrates a downstream application of "
            "the super-resolved product."
        )


# ============================================================
# TRUST / UNCERTAINTY
# ============================================================

elif view == "🛡️ Trust & Uncertainty":

    st.header("🛡️ Trust & Uncertainty")

    st.markdown(
        """
        An empirical reliability framework combines observation
        consistency, spatial stability and input validity.
        """
    )

    col1, col2 = st.columns(2)

    with col1:

        fig, ax = plt.subplots(figsize=(9, 7))

        im = ax.imshow(
            trust,
            cmap="RdYlGn",
            vmin=0,
            vmax=100
        )

        ax.set_title("Trust Score — 2.5 m")
        ax.axis("off")

        fig.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04,
            label="Trust"
        )

        st.pyplot(fig)
        plt.close(fig)

    with col2:

        fig, ax = plt.subplots(figsize=(9, 7))

        im = ax.imshow(
            uncertainty,
            cmap="magma",
            vmin=0,
            vmax=100
        )

        ax.set_title("Uncertainty — 2.5 m")
        ax.axis("off")

        fig.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04,
            label="Uncertainty"
        )

        st.pyplot(fig)
        plt.close(fig)

    st.divider()

    tv = metrics["trust_validation"]

    c1, c2, c3 = st.columns(3)

    c1.metric("Mean Trust", f'{tv["mean_trust"]:.2f}')
    c2.metric("Median Trust", f'{tv["median_trust"]:.2f}')
    c3.metric("Mean Uncertainty", f'{tv["mean_uncertainty"]:.2f}')

    st.info(
        "Trust is an empirical reliability indicator, "
        "not a calibrated probability of correctness."
    )


# ============================================================
# VALIDATION
# ============================================================

elif view == "📋 Validation Metrics":

    st.header("📋 Validation Results")

    sen = metrics["sen2srlite"]
    tv = metrics["trust_validation"]

    st.subheader("SEN2SRLite")

    c1, c2, c3 = st.columns(3)

    c1.metric("SAM Mean", f'{sen["sam_mean_deg"]:.4f}°')
    c2.metric("SAM Median", f'{sen["sam_median_deg"]:.4f}°')
    c3.metric("SAM RMSE", f'{sen["sam_rmse_deg"]:.4f}°')

    st.divider()

    st.subheader("NDVI")

    c1, c2, c3 = st.columns(3)

    c1.metric("MAE", f'{sen["ndvi_mae"]:.5f}')
    c2.metric("RMSE", f'{sen["ndvi_rmse"]:.5f}')
    c3.metric(
        "Correlation",
        f'{sen["ndvi_correlation"]:.4f}'
    )

    st.divider()

    st.subheader("Trust Validation")

    table = {
        "Measurement": [
            "Mean Trust",
            "Median Trust",
            "Mean Uncertainty",
            "Trust/Error Correlation",
            "High Trust",
            "Medium Trust",
            "Low Trust",
            "Very Low Trust"
        ],
        "Value": [
            tv["mean_trust"],
            tv["median_trust"],
            tv["mean_uncertainty"],
            tv["trust_error_correlation"],
            tv["high_trust_percent"],
            tv["medium_trust_percent"],
            tv["low_trust_percent"],
            tv["very_low_trust_percent"]
        ]
    }

    st.dataframe(
        table,
        use_container_width=True,
        hide_index=True
    )


# ============================================================
# DOWNLOADS
# ============================================================

st.divider()

st.header("📥 Download Products")

download_files = {
    "SEN2SR 2.5 m": SR_FILE,
    "Bicubic 2.5 m": BICUBIC_FILE,
    "NDVI": NDVI_FILE,
    "Trust Map": TRUST_FILE,
    "Uncertainty Map": UNCERTAINTY_FILE,
    "Metrics": METRICS_FILE
}

cols = st.columns(3)

for i, (label, path) in enumerate(download_files.items()):

    if path.exists():

        with open(path, "rb") as f:
            data = f.read()

        mime = (
            "application/json"
            if path.suffix == ".json"
            else "image/tiff"
        )

        cols[i % 3].download_button(
            f"⬇️ {label}",
            data=data,
            file_name=path.name,
            mime=mime
        )


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "SIH 2026 | Problem Statement 26142 | "
    "Deep Learning Based Super Resolution Mapping"
)
'''

APP.write_text(app_code, encoding="utf-8")

print("✅ app.py created!")
print(APP)
print("Size:", APP.stat().st_size, "bytes")

✅ app.py created!
/content/drive/MyDrive/SIH26142_SuperResolution/dashboard/app.py
Size: 13078 bytes


In [ ]:
!pkill -f streamlit || true

^C


In [ ]:
!ss -ltnp | grep 8501 || echo "✅ Port 8501 is free"

✅ Port 8501 is free


run from here


In [ ]:
import subprocess
import time
import os

APP = "/content/drive/MyDrive/SIH26142_SuperResolution/dashboard/app.py"

# Stop any old Streamlit processes
os.system("pkill -f 'streamlit run' || true")

# Start Streamlit in background
streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", APP,
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("🚀 Streamlit started!")
print("PID:", streamlit_process.pid)

time.sleep(5)

# Check whether it is alive
if streamlit_process.poll() is None:
    print("✅ Streamlit process is still running")
else:
    print("❌ Streamlit stopped unexpectedly")

🚀 Streamlit started!
PID: 3386
✅ Streamlit process is still running


In [ ]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Sat, 19 Sep 2026 09:43:19 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7260
last-modified: Sat, 19 Sep 2026 09:37:30 GMT
etag: "290150c732b3d7be9dc58edbf6a85b13"
cache-control: no-cache



In [ ]:
!pkill -f cloudflared || true

^C


In [ ]:
import subprocess
import time

cloudflare_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8501",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("🚀 Starting Cloudflare tunnel...\n")

for _ in range(40):
    line = cloudflare_process.stdout.readline()
    if line:
        print(line, end="")
    time.sleep(0.2)

🚀 Starting Cloudflare tunnel...

2026-09-19T09:43:25Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-19T09:43:25Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-19T09:43:27Z INF +--------------------------------------------------------------------------------------------+
2026-09-19T09:43:27Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-19T09:43:27Z INF |  https://milk-adminis